In [ ]:
# Install Required Libraries
!pip install pypdf2 PyMuPDF pdfplumber nltk

# Install OCR dependencies
!apt-get update
!apt-get install -y poppler-utils tesseract-ocr
!pip install pytesseract pdf2image

import json
import os
import re
import nltk
from nltk.tokenize import sent_tokenize

# Download required NLTK data
nltk.download('punkt_tab')
nltk.download('punkt')

# ================================================
# SECTION 1: PROCESS ALL CONSTITUTION PDFs WITH OCR FALLBACK
# ================================================

INPUT_DIR = "/content/constitutions"
OUTPUT_JSON = "/content/processed_constitutions.json"

constitutions_data = {}

def extract_text_with_ocr(pdf_path):
    """Extract text from PDF using OCR as fallback for image-based PDFs"""
    text = ""

    # Method 1: Try PyMuPDF first (fitz) - fastest
    try:
        import fitz
        doc = fitz.open(pdf_path)
        for page in doc:
            text += page.get_text() + "\n"
        doc.close()
        if text.strip():  # If we got text
            return text, "PyMuPDF"
    except Exception as e:
        print(f"  PyMuPDF failed: {e}")
        text = ""

    # Method 2: Try pdfplumber
    if not text.strip():
        try:
            import pdfplumber
            with pdfplumber.open(pdf_path, strict=False) as pdf:
                for page in pdf.pages:
                    t = page.extract_text()
                    if t:
                        text += t + "\n"
            if text.strip():
                return text, "pdfplumber"
        except Exception as e:
            print(f"  pdfplumber failed: {e}")
            text = ""

    # Method 3: Try OCR for image-based PDFs
    if not text.strip():
        try:
            print(f"  Attempting OCR extraction...")
            from pdf2image import convert_from_path
            import pytesseract

            # Convert PDF to images
            images = convert_from_path(pdf_path)

            # Extract text from each image
            ocr_text = ""
            for i, image in enumerate(images):
                page_text = pytesseract.image_to_string(image)
                ocr_text += f"--- Page {i+1} ---\n{page_text}\n"

            if ocr_text.strip():
                return ocr_text, "OCR (Tesseract)"
            else:
                return "", "OCR failed - no text"

        except Exception as e:
            print(f"  OCR failed: {e}")
            return "", f"OCR error: {str(e)}"

    return text, "No text extracted"

# Process each PDF file
for file_name in os.listdir(INPUT_DIR):
    if file_name.endswith(".pdf"):
        full_path = os.path.join(INPUT_DIR, file_name)
        print(f"\nProcessing: {file_name}")

        try:
            raw_text, method = extract_text_with_ocr(full_path)

            print(f"  Used method: {method}")

            if not raw_text or not raw_text.strip():
                print(f"  WARNING: No text extracted from {file_name}")
                raw_text = ""  # Set to empty string

            # Tokenize sentences
            sentences = sent_tokenize(raw_text) if raw_text else []

            constitutions_data[file_name] = {
                "file_name": file_name,
                "raw_text": raw_text,
                "sentence_count": len(sentences),
                "sentences": sentences,
                "extraction_method": method
            }

            print(f"  Success: Extracted {len(sentences)} sentences")

        except Exception as e:
            print(f"  ERROR processing {file_name}: {str(e)}")
            constitutions_data[file_name] = {
                "file_name": file_name,
                "error": str(e),
                "raw_text": "",
                "sentences": [],
                "extraction_method": "Error"
            }

# Save to JSON file
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(constitutions_data, f, indent=4, ensure_ascii=False)

print(f"\n{'='*50}")
print(f"PROCESSING COMPLETE")
print(f"{'='*50}")
print(f"Output saved to: {OUTPUT_JSON}")
print(f"Total files processed: {len(constitutions_data)}")

# Show summary
print(f"\nExtraction Summary:")
for file_name, data in constitutions_data.items():
    method = data.get('extraction_method', 'Unknown')
    sentences = data.get('sentence_count', 0)
    status = "✓" if sentences > 0 else "✗"
    print(f"  {status} {file_name}: {sentences} sentences ({method})")

successful = sum(1 for data in constitutions_data.values() if data.get('sentence_count', 0) > 0)
failed = len(constitutions_data) - successful
print(f"\nSuccessful extractions: {successful}/{len(constitutions_data)}")
print(f"Failed extractions: {failed}/{len(constitutions_data)}")

# ================================================
# SECTION 2: EXTRACT ARTICLES FROM MAIN CONSTITUTION
# ================================================

print(f"\n{'='*50}")
print("EXTRACTING CONSTITUTION ARTICLES")
print(f"{'='*50}")

# Find the main constitution file (usually the latest one)
main_constitution = None
for file_name in constitutions_data:
    if "constitution" in file_name.lower() and constitutions_data[file_name]['sentence_count'] > 0:
        if main_constitution is None or "latest" in file_name.lower():
            main_constitution = file_name

if main_constitution:
    print(f"Using main constitution file: {main_constitution}")
    constitution_raw = constitutions_data[main_constitution]['raw_text']
else:
    # Fallback to first file with text
    for file_name, data in constitutions_data.items():
        if data['sentence_count'] > 0:
            constitution_raw = data['raw_text']
            main_constitution = file_name
            print(f"Using file: {main_constitution}")
            break

print(f"Text length: {len(constitution_raw)} characters")

# ================================================
# SECTION 3: SMART ARTICLE EXTRACTION
# ================================================

def extract_articles_smartly(text, file_name):
    """Smart article extraction that adapts to different formats"""
    print(f"\nExtracting articles from: {file_name}")

    articles = {}

    # Strategy 1: Look for standard constitution format
    # Find where the actual articles start (after table of contents)
    lines = text.split('\n')

    # Look for chapter markers
    chapter_markers = ['CHAPTER', 'Chapter', 'chapter']
    article_start = 0

    for i, line in enumerate(lines):
        if any(marker in line for marker in chapter_markers):
            # Check if this is a real chapter (has roman numerals or numbers)
            if re.search(r'CHAPTER\s+[IVXLCDM]+', line) or re.search(r'CHAPTER\s+\d+', line):
                article_start = i
                break

    if article_start > 0:
        print(f"Found chapter start at line {article_start}")
        text = '\n'.join(lines[article_start:])

    # Strategy 2: Clean and prepare text
    text = re.sub(r'\s+', ' ', text)  # Normalize whitespace

    # Strategy 3: Try different extraction patterns based on file type
    patterns = [
        # Pattern for "1. Text" or "1 Text" (most common)
        r'(?:^|\s)(\d+[A-Z]?)\.?\s+(.*?)(?=(?:\s\d+[A-Z]?\.?\s|CHAPTER|Article|$))',
        # Pattern for "Article 1" format
        r'Article\s+(\d+[A-Z]?)\s+(.*?)(?=(?:Article\s+\d+[A-Z]?|CHAPTER|$))',
        # Pattern for numbered sections with parentheses
        r'(\d+[A-Z]?)\.\s*\((.*?)\)',
    ]

    all_matches = []
    best_pattern = ""

    for pattern in patterns:
        matches = list(re.finditer(pattern, text, re.IGNORECASE | re.DOTALL))
        if len(matches) > len(all_matches):
            all_matches = matches
            best_pattern = pattern

    print(f"Best pattern found {len(all_matches)} matches")

    # Extract articles
    for match in all_matches:
        article_num = match.group(1).strip()
        article_text = match.group(2).strip()

        # Clean and validate
        article_text = re.sub(r'\s+', ' ', article_text)

        # Skip if too short or looks like noise
        if len(article_text) > 20 and not re.search(r'\.\.\.|…|Page \d+', article_text):
            # Extract meaningful keywords
            keywords = list(set(re.findall(r'\b[A-Z][a-z]+\b', article_text)))

            # Filter out common words
            common_words = {'The', 'And', 'For', 'With', 'This', 'That', 'Which'}
            keywords = [kw for kw in keywords if kw not in common_words][:20]

            articles[f"Article{article_num}"] = {
                "article_number": article_num,
                "text": article_text[:5000],
                "keywords": keywords,
                "source_file": file_name
            }

    return articles

# Extract articles from all constitution files
all_articles_combined = {}

for file_name, data in constitutions_data.items():
    if data['sentence_count'] > 0 and "constitution" in file_name.lower():
        print(f"\n{'='*50}")
        print(f"Processing: {file_name}")
        print(f"{'='*50}")

        articles = extract_articles_smartly(data['raw_text'], file_name)

        # Merge articles (avoid duplicates, keep from latest constitution)
        for article_key, article_data in articles.items():
            if article_key not in all_articles_combined:
                all_articles_combined[article_key] = article_data
            else:
                # If duplicate, keep the one with longer text (likely more complete)
                if len(article_data['text']) > len(all_articles_combined[article_key]['text']):
                    all_articles_combined[article_key] = article_data

print(f"\n{'='*50}")
print(f"COMBINED EXTRACTION RESULTS")
print(f"{'='*50}")
print(f"Total unique articles extracted: {len(all_articles_combined)}")

if all_articles_combined:
    # Sort by article number
    def get_article_num(key):
        match = re.search(r'Article(\d+)', key)
        return int(match.group(1)) if match else 0

    sorted_keys = sorted(all_articles_combined.keys(), key=get_article_num)

    print(f"\nFirst 10 articles:")
    for key in sorted_keys[:10]:
        article = all_articles_combined[key]
        preview = article['text'][:80].replace('\n', ' ')
        print(f"  {key}: {preview}...")

    print(f"\nFundamental Rights Articles (10-18):")
    for num in range(10, 19):
        key = f"Article{num}"
        if key in all_articles_combined:
            article = all_articles_combined[key]
            preview = article['text'][:60].replace('\n', ' ')
            print(f"  {key}: {preview}...")
        else:
            print(f"  {key}: NOT FOUND")

    # Show article count by source
    print(f"\nArticles by source file:")
    source_counts = {}
    for article in all_articles_combined.values():
        source = article['source_file']
        source_counts[source] = source_counts.get(source, 0) + 1

    for source, count in source_counts.items():
        print(f"  {source}: {count} articles")

# ================================================
# SECTION 4: SAVE ALL ARTICLES
# ================================================

# Save comprehensive articles JSON
articles_output_file = "constitution_all_articles_COMPLETE.json"
with open(articles_output_file, "w", encoding="utf-8") as f:
    json.dump(all_articles_combined, f, indent=4, ensure_ascii=False)

print(f"\nAll articles saved to: {articles_output_file}")

# ================================================
# SECTION 5: CREATE DETECTION PATTERNS
# ================================================

def create_comprehensive_patterns(articles_dict):
    """Create patterns for all extracted articles"""
    patterns = {}

    if not articles_dict:
        print("No articles to create patterns from!")
        return patterns

    # Common article descriptions
    article_descriptions = {
        "1": "The State",
        "2": "Unitary State",
        "3": "Sovereignty of the People",
        "4": "Exercise of Sovereignty",
        "5": "Territory of the Republic",
        "6": "National Flag",
        "7": "National Anthem",
        "8": "National Day",
        "9": "Buddhism",
        "10": "Freedom of thought, conscience and religion",
        "11": "Freedom from torture",
        "12": "Right to equality",
        "13": "Freedom from arbitrary arrest",
        "14": "Freedom of speech, assembly, association",
        "15": "Freedom of movement and residence",
        "16": "Language rights",
        "17": "Existing law",
        "18": "Interpretation",
        "30": "President of the Republic",
        "105": "Establishment of Courts",
        "148": "Consolidated Fund"
    }

    for article_key, article_data in articles_dict.items():
        try:
            article_num = article_data["article_number"]
            text = article_data["text"].lower()
            source = article_data["source_file"]

            # Get description
            description = article_descriptions.get(article_num, f"Article {article_num}")

            # Extract key terms
            words = re.findall(r'\b[a-z]{4,}\b', text)
            word_freq = {}
            for word in words:
                word_freq[word] = word_freq.get(word, 0) + 1

            # Get top keywords (excluding common words)
            common_words = {'shall', 'may', 'such', 'every', 'person', 'state',
                           'republic', 'lanka', 'president', 'parliament', 'court',
                           'government', 'minister', 'power', 'right'}

            top_words = []
            for word, freq in sorted(word_freq.items(), key=lambda x: x[1], reverse=True):
                if word not in common_words and len(top_words) < 5:
                    top_words.append(word)

            # Create pattern
            if top_words:
                pattern_terms = "|".join([re.escape(word) for word in top_words])
                patterns[article_num] = {
                    "pattern": f"({pattern_terms})",
                    "description": description,
                    "keywords": top_words,
                    "source": source,
                    "text_preview": article_data["text"][:100]
                }

        except Exception as e:
            print(f"Error creating pattern for {article_key}: {e}")
            continue

    return patterns

# Create patterns
all_patterns = create_comprehensive_patterns(all_articles_combined)

# Save patterns to JSON
patterns_output_file = "constitution_detection_patterns_COMPLETE.json"
with open(patterns_output_file, "w", encoding="utf-8") as f:
    json.dump(all_patterns, f, indent=4, ensure_ascii=False)

print(f"\nDetection patterns saved to: {patterns_output_file}")
print(f"Created patterns for {len(all_patterns)} articles")

# ================================================
# SECTION 6: FINAL VALIDATION REPORT
# ================================================

print(f"\n{'='*50}")
print("FINAL VALIDATION REPORT")
print(f"{'='*50}")

print(f"\n1. PDF EXTRACTION STATUS (ALL FILES):")
print(f"{'-'*40}")

all_extracted = True
for file_name, data in constitutions_data.items():
    status = "✓ SUCCESS" if data['sentence_count'] > 0 else "✗ FAILED"
    method = data.get('extraction_method', 'Unknown')
    sentences = data['sentence_count']

    if data['sentence_count'] == 0:
        all_extracted = False

    print(f"  {status:12} {file_name:60} {sentences:5} sentences ({method})")

print(f"\n2. ARTICLE EXTRACTION SUMMARY:")
print(f"{'-'*40}")
print(f"  Total unique articles: {len(all_articles_combined)}")

if all_articles_combined:
    # Count by number ranges
    fundamental = sum(1 for key in all_articles_combined.keys()
                     if re.search(r'Article(1[0-8])\b', key))
    early = sum(1 for key in all_articles_combined.keys()
               if re.search(r'Article([1-9]|[1-9][0-9]|1[0-7][0-9])\b', key))

    print(f"  Fundamental Rights (10-18): {fundamental}/9")
    print(f"  Articles 1-179: {early}")

    # Check key articles
    print(f"\n3. KEY ARTICLES CHECK:")
    print(f"{'-'*40}")

    key_articles = {
        "Essential": ["1", "2", "3", "4", "5"],
        "Fundamental Rights": ["10", "11", "12", "13", "14"],
        "Government": ["30", "42", "105", "148"]
    }

    for category, articles in key_articles.items():
        found = []
        missing = []
        for art in articles:
            if f"Article{art}" in all_articles_combined:
                found.append(art)
            else:
                missing.append(art)

        if found:
            print(f"  ✓ {category}: Found {len(found)}/{len(articles)}")
        if missing:
            print(f"  ✗ {category}: Missing {len(missing)}/{len(articles)}")

print(f"\n4. GENERATED FILES:")
print(f"{'-'*40}")
print(f"  {OUTPUT_JSON}: All PDF texts with extraction methods")
print(f"  {articles_output_file}: {len(all_articles_combined)} unique articles")
print(f"  {patterns_output_file}: {len(all_patterns)} detection patterns")

print(f"\n5. OCR STATUS FOR FAILED FILES:")
print(f"{'-'*40}")
failed_files = [f for f, d in constitutions_data.items() if d['sentence_count'] == 0]
if failed_files:
    print(f"  Failed files: {len(failed_files)}")
    for file in failed_files:
        print(f"    - {file}")
    print(f"\n  Note: These appear to be image-based/scanned PDFs.")
    print(f"        OCR was attempted but may need additional configuration.")
else:
    print(f"  ✓ All files successfully extracted!")

# ================================================
# SECTION 7: DOWNLOAD ALL FILES
# ================================================

from google.colab import files

print(f"\n{'='*50}")
print("DOWNLOADING ALL FILES")
print(f"{'='*50}")

files_to_download = [
    OUTPUT_JSON,
    articles_output_file,
    patterns_output_file
]

for file_path in files_to_download:
    if os.path.exists(file_path):
        print(f"\nDownloading: {file_path}")
        files.download(file_path)
    else:
        print(f"\nWarning: File not found: {file_path}")

print(f"\n{'='*50}")
print("PROCESSING COMPLETE!")
print(f"{'='*50}")

if all_extracted:
    print(f"\n✅ SUCCESS: All PDFs were successfully extracted!")
else:
    print(f"\n⚠ PARTIAL SUCCESS: Some PDFs failed extraction (likely image-based).")
    print(f"   Successfully extracted: {successful}/{len(constitutions_data)} files")
    print(f"   Failed: {failed}/{len(constitutions_data)} files")

print(f"\n📁 Files downloaded:")
print(f"   1. {OUTPUT_JSON} - All PDF texts with extraction methods")
print(f"   2. {articles_output_file} - {len(all_articles_combined)} constitution articles")
print(f"   3. {patterns_output_file} - Detection patterns for articles")

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:3 https://cli.github.com/packages stable InRelease
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [6,281 kB]
Get:13 https://developer.download.nvidia.com/compute/cuda/repos/ubu

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!



Processing: constitution-Latest.pdf
  Used method: PyMuPDF
  Success: Extracted 1983 sentences

Processing: 1978ConstitutionWithoutAmendments.pdf
  Used method: PyMuPDF
  Success: Extracted 1031 sentences

Processing: Report-of-The-Donoughmore-Commission-gbdoed.pdf
  Used method: PyMuPDF
  Success: Extracted 3103 sentences

Processing: 1972-Constitution-of-Sri-Lanka-Article-105-–134-Chapter-XIII-dked1t.pdf
  pdfplumber failed: PDF.open() got an unexpected keyword argument 'strict'
  Attempting OCR extraction...
  Used method: OCR (Tesseract)
  Success: Extracted 693 sentences

Processing: Report-of-The-Soulbury-Commission-hvpase.pdf
  Used method: PyMuPDF
  Success: Extracted 3851 sentences

PROCESSING COMPLETE
Output saved to: /content/processed_constitutions.json
Total files processed: 5

Extraction Summary:
  ✓ constitution-Latest.pdf: 1983 sentences (PyMuPDF)
  ✓ 1978ConstitutionWithoutAmendments.pdf: 1031 sentences (PyMuPDF)
  ✓ Report-of-The-Donoughmore-Commission-gbdoed.pdf: 31

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>


Downloading: constitution_detection_patterns_COMPLETE.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


PROCESSING COMPLETE!

✅ SUCCESS: All PDFs were successfully extracted!

📁 Files downloaded:
   1. /content/processed_constitutions.json - All PDF texts with extraction methods
   2. constitution_all_articles_COMPLETE.json - 371 constitution articles
   3. constitution_detection_patterns_COMPLETE.json - Detection patterns for articles
